# 교통사고 과실비율 Video QA — 파일럿 실험 노트북

**실험 설계 문서 Phase 1 파일럿** — 클래스별 10개 샘플로 하이퍼파라미터를 탐색합니다.

---

## 빠른 시작
1. **런타임 > GPU 변경** → A100 (또는 L4) 선택
2. **Cell 1** 실행 → Google Drive 마운트
3. **⚙️ CONFIG 셀**에서 `EXPERIMENT_NAME` 과 `PROJECT_ROOT` 만 수정
4. **런타임 > 모두 실행** (`Ctrl+F9`)

---

## 실험 카탈로그

| 계열 | 실험 이름 | 변수 |
|------|----------|------|
| Phase 0 (베이스라인) | `phase0_no_prompt` `phase0_default_prompt` | 학습 없음 |
| Phase 1A LoRA rank | `A-1` `A-2` `A-3` `A-4` | rank 8 / 16 / 32 / 64 |
| Phase 1B fps/해상도 | `B-1` `B-2` `B-3` | fps 0.5 / 1.0 / 2.0 |
| Phase 1C 프롬프트 | `C-1` `C-2` `C-3` `C-4` | 없음 / 기본 / 코드힌트 / 법률 |
| Phase 2 본학습 | `phase2_main` | 전체 데이터 soft-capping |

---

## 평가 지표 우선순위
1. `fault_ratio` **Exact Match** — 핵심 비즈니스 지표
2. 전체 **ROUGE-L 평균** — 서술형 답변 품질
3. `fault_compare` **Accuracy** — 법적 판단 능력

In [ ]:
# ── Google Drive 마운트 ───────────────────────────────────────
# 패키지 소스를 Drive에서 복사하기 위해 마운트가 필요합니다.
# 데이터(비디오)는 HTTP 서버에서 직접 내려받으므로 Drive IO에 의존하지 않습니다.
import os

IS_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(globals())

MOUNT_DRIVE = True   # 패키지 설치 시 Drive 복사 필요 — False로 바꾸면 pip 설치 셀을 건너뜁니다

if IS_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive 마운트 완료')
else:
    print('Drive 마운트 건너뜀')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           ⚙️  실험 설정  —  이 셀만 수정하세요             ║
# ╚══════════════════════════════════════════════════════════════╝

# ─── 0. Drive 경로 — 패키지 소스 설치에만 사용 ───────────────
# (데이터 접근은 HTTP 서버 직접 다운로드이므로 이 경로로 파일을 읽지 않습니다)
DRIVE_ROOT = '/content/drive/MyDrive/멋쟁이사자처럼/traffic-fault-platform'

# ─── 1. 데이터 서버 URL ────────────────────────────────────────
BASE_URL    = 'https://data.taeo-dev.com/dataset/traffic/1.Training'
RAW_URL     = f'{BASE_URL}/raw_data_231108_add'
LABEL_URL   = f'{BASE_URL}/label_data_231108_add'
QA_JSON_URL = 'https://data.taeo-dev.com/dataset/traffic/qa_dataset.json'

# ─── 2. 실험 선택 ─────────────────────────────────────────────
#   Phase 0 (베이스라인, 학습 없음)
#     'phase0_no_prompt'        — 시스템 프롬프트 없음
#     'phase0_default_prompt'   — 기본 시스템 프롬프트
#
#   Phase 1A — LoRA rank 탐색
#     'A-1'  rank=8    'A-2'  rank=16 (기본)
#     'A-3'  rank=32   'A-4'  rank=64
#
#   Phase 1B — 입력 해상도 / fps 탐색
#     'B-1'  fps=0.5, 320×320
#     'B-2'  fps=1.0, 360×420 (기본)
#     'B-3'  fps=2.0, 360×420
#
#   Phase 1C — 프롬프트 전략
#     'C-1'  없음 (ablation)   'C-2'  기본값
#     'C-3'  case_code 힌트    'C-4'  도로교통법 주입
#
#   Phase 2 — 본 학습
#     'phase2_main'  전체 데이터, soft capping 200
EXPERIMENT_NAME = 'A-2'

# ─── 3. 파일럿 샘플링 ────────────────────────────────────────
PILOT_SAMPLES_PER_CLASS = 10

# ─── 4. 하이퍼파라미터 오버라이드 ────────────────────────────
# None 이면 카탈로그(ExperimentConfig) 또는 TrainingConfig 기본값 사용
# 값을 입력하면 기본값을 덮어씁니다
OVERRIDE = {
    # ── ExperimentConfig 필드 (카탈로그 기본값 오버라이드) ────
    'lora_r':                     None,   # LoRA rank          기본: 8
    'lora_alpha':                 None,   # LoRA alpha         기본: 16
    'lora_dropout':               None,   # LoRA dropout       기본: 0.05
    'learning_rate':              None,   # 학습률             기본: 2e-4
    'fps':                        None,   # 초당 프레임 수     기본: 0.5
    'max_pixels':                 None,   # 최대 픽셀 수       기본: 320*320=102400
    'num_train_epochs':           None,   # 학습 에폭 수       기본: 5
    'system_prompt_key':          None,   # 시스템 프롬프트 키 기본: 'default'
    'eval_steps':                 None,   # 평가 주기(스텝)    기본: 100
    # ── TrainingConfig 전용 필드 ──────────────────────────────
    'gradient_accumulation_steps': None,  # 그래디언트 누적    기본: 4
    'dataloader_num_workers':      None,  # 데이터로더 워커 수 기본: 2
    'save_steps':                  None,  # 체크포인트 저장 주기 기본: 200
}

# ─── 5. 학습 제어 ─────────────────────────────────────────────
SMOKE_TEST = False
SEED       = 42

# ─── 로컬 SSD 경로 (수정 불필요) ─────────────────────────────
LOCAL_ROOT      = '/content/local_data'   # aria2c 다운로드 목적지
CHECKPOINT_ROOT = '/content/checkpoints'
OUTPUT_DIR      = '/content/outputs'

# 학습 시 읽을 로컬 경로 (download_pilot_to_ssd 실행 후 채워짐)
VIDEO_ROOT       = f'{LOCAL_ROOT}/raw'
LABEL_ROOT       = f'{LOCAL_ROOT}/label'
QA_JSON_PATH     = f'{LOCAL_ROOT}/qa_dataset.json'
VIDEO_CACHE_DIR  = f'{LOCAL_ROOT}/frame_cache'  # 비디오당 1회만 디코딩, ~3.6GB

print(f'실험       : {EXPERIMENT_NAME}')
print(f'파일럿 상한: {PILOT_SAMPLES_PER_CLASS}개/클래스')
print(f'RAW_URL    : {RAW_URL}')
print(f'LOCAL_ROOT : {LOCAL_ROOT}')
print(f'DRIVE_ROOT : {DRIVE_ROOT}')

In [ ]:
# ── 패키지 설치 (Colab 전용) ──────────────────────────────────
if IS_COLAB:
    import subprocess, sys, shutil
    from pathlib import Path

    # 1. PyPI 의존성 설치
    pkgs = [
        'transformers>=4.51',
        'peft>=0.11',
        'accelerate>=0.30',
        'safetensors>=0.4',
        'qwen-vl-utils>=0.0.8',
        'av',
        'decord',
        'beautifulsoup4',   # HTTP 디렉토리 파싱
        'requests',
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)

    # aria2c (병렬 다운로더)
    subprocess.check_call(['apt-get', 'install', '-y', '-q', 'aria2'])

    # 2. 로컬 패키지: 한글 경로 문제 해결
    #    Drive 경로에 한글이 포함되어 pip install -e 가 실패하므로
    #    ASCII 경로(/content/pkgs/)로 복사한 뒤 설치합니다.
    PKG_LOCAL = Path('/content/pkgs')
    PKG_LOCAL.mkdir(parents=True, exist_ok=True)

    for src_rel in ['packages/ai-core', 'apps/training-runner']:
        src = Path(DRIVE_ROOT) / src_rel
        dst = PKG_LOCAL / src.name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(str(src), str(dst))
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-e', str(dst), '-q'],
            capture_output=True, text=True,
        )
        status = '✓' if result.returncode == 0 else result.stderr.strip()
        print(f'{dst.name}: {status}')

    print('패키지 설치 완료')
else:
    print('로컬 환경 — 패키지 설치 건너뜀 (venv 사용 권장)')

In [ ]:
# ── 임포트 ────────────────────────────────────────────────────
# 로컬 패키지(ai-core, training-runner)는 cell-install 에서
# pip install -e 로 설치했으므로 sys.path 조작 없이 바로 임포트합니다.
import sys
import json
import random
import time
from pathlib import Path
from collections import defaultdict

import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# 한글 폰트 설정 (Colab)
if IS_COLAB:
    import subprocess
    subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-nanum'], capture_output=True)
    import matplotlib.font_manager as fm
    fm._load_fontmanager(try_read_cache=False)
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False

from training_runner.configs import (
    TrainingConfig, ExperimentConfig,
    EXPERIMENT_CATALOG, SYSTEM_PROMPTS,
    get_experiments_by_phase, list_experiments,
)
from training_runner.evaluation.metrics import format_summary
from training_runner.evaluation.evaluator import EvaluatorConfig, run_evaluation

print('임포트 완료')

In [ ]:
# ── GPU 확인 + 실험 설정 검증 ──────────────────────────────────
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU  : {gpu_name}')
    print(f'VRAM : {vram_gb:.1f} GB')
    if vram_gb < 16:
        print('[WARNING] VRAM < 16GB — fps/max_pixels를 낮춰야 OOM을 피할 수 있습니다.')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print('MPS (Apple Silicon) 감지')
else:
    print('[WARNING] GPU 없음 — 학습이 매우 느립니다. 런타임 유형을 GPU로 변경하세요.')

# 실험 검증
if EXPERIMENT_NAME not in EXPERIMENT_CATALOG:
    available = list(EXPERIMENT_CATALOG.keys())
    raise ValueError(f'알 수 없는 실험: {EXPERIMENT_NAME!r}. 사용 가능: {available}')

exp_info = EXPERIMENT_CATALOG[EXPERIMENT_NAME]
print(f'\n실험 : {exp_info.name}  (Phase {exp_info.phase})')
print(f'설명 : {exp_info.description}')
print(f'학습 : {"건너뜀 (Phase 0 베이스라인)" if exp_info.skip_training else "진행"}')

print('\n[등록된 전체 실험 목록]')
list_experiments()

## 데이터 탐색

클래스(case_code)별 비디오 수를 확인하고 **파일럿 샘플링(10개/클래스)** 결과를 시각화합니다.

In [ ]:
# ── HTTP 서버에서 클래스별 비디오 파일 수 탐색 ───────────────
import requests
from bs4 import BeautifulSoup

def list_http_dir(url: str) -> list[str]:
    """HTTP 디렉토리 인덱스에서 파일/폴더 이름 목록 반환."""
    resp = requests.get(url.rstrip('/') + '/', timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    return [
        a['href'].rstrip('/')
        for a in soup.find_all('a', href=True)
        if not a['href'].startswith(('?', '/', 'http')) and a['href'] != '../'
    ]

# 클래스 목록 및 비디오 수 파싱
class_counts: dict = {}
for cls in sorted(list_http_dir(RAW_URL)):
    files = [f for f in list_http_dir(f'{RAW_URL}/{cls}') if f.endswith('.mp4')]
    if files:
        class_counts[cls] = len(files)

n_classes    = len(class_counts)
total_videos = sum(class_counts.values())

print('=' * 50)
print('데이터 현황 (HTTP 서버)')
print('=' * 50)
print(f'  클래스 수     : {n_classes}')
print(f'  전체 비디오   : {total_videos}')
print(f'  클래스당 평균 : {total_videos / n_classes:.1f}개')
print(f'  최소 / 최대   : {min(class_counts.values())} / {max(class_counts.values())}개')

# QA 데이터셋 확인
import requests as _req
qa_data = _req.get(QA_JSON_URL, timeout=30).json()
qa_stems = {Path(e['video_id']).stem for e in qa_data}
video_stems = set()
for cls, cnt in class_counts.items():
    pass  # stem 목록은 sampling 단계에서 구성
print(f'\nQA 항목   : {len(qa_data)}')
print(f'QA 쌍 총계: {sum(len(e["qa_pairs"]) for e in qa_data)}')

# ── 분포 시각화 ──────────────────────────────────────────────
cap = PILOT_SAMPLES_PER_CLASS or 9999
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
fig.suptitle('클래스별 비디오 수 분포 (HTTP 서버)', fontsize=13, fontweight='bold')

counts_sorted  = sorted(class_counts.values())
axes[0].hist(counts_sorted, bins=min(20, n_classes), color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(cap, color='crimson', linestyle='--', lw=2, label=f'파일럿 상한 ({cap}개)')
axes[0].set_xlabel('비디오 수'); axes[0].set_ylabel('클래스 수')
axes[0].set_title('원본 분포'); axes[0].legend()

classes_sorted = sorted(class_counts.keys())
after_counts   = [min(class_counts[c], cap) for c in classes_sorted]
colors = ['orange' if class_counts[c] < cap else 'steelblue' for c in classes_sorted]
axes[1].bar(range(len(classes_sorted)), after_counts, color=colors, alpha=0.85, width=1.0)
axes[1].set_xlabel('클래스 인덱스'); axes[1].set_ylabel('샘플 수')
axes[1].set_title(f'파일럿 샘플링 후 (최대 {cap}개/클래스)')
axes[1].legend(handles=[
    mpatches.Patch(color='steelblue', label=f'≥{cap}개 (상한 적용)'),
    mpatches.Patch(color='orange',    label=f'<{cap}개 (전부 사용)'),
])
plt.tight_layout(); plt.show()

n_capped = sum(1 for v in class_counts.values() if v >= cap)
print(f'상한 적용 클래스: {n_capped}/{n_classes}  |  샘플링 후 예상 비디오: {sum(after_counts)}')


In [ ]:
# ══════════════════════════════════════════════════════════════
# 파일럿 샘플링 — 클래스별 최대 PILOT_SAMPLES_PER_CLASS개 선택
# ══════════════════════════════════════════════════════════════

def sample_pilot_data(
    raw_url: str,
    max_per_class: int = 10,
    seed: int = 42,
) -> tuple:
    """HTTP 서버 디렉토리 인덱스를 파싱해 클래스별 비디오 stem을 샘플링합니다."""
    rng = random.Random(seed)
    sampled_stems:       set  = set()
    class_sample_counts: dict = {}

    for cls in sorted(list_http_dir(raw_url)):
        videos = sorted(f for f in list_http_dir(f'{raw_url}/{cls}') if f.endswith('.mp4'))
        if not videos:
            continue
        n      = min(max_per_class, len(videos)) if max_per_class else len(videos)
        picked = rng.sample(videos, n)
        class_sample_counts[cls] = n
        for fname in picked:
            sampled_stems.add(Path(fname).stem)

    return sampled_stems, class_sample_counts


# ── 샘플링 실행 ──────────────────────────────────────────────
pilot_stems, pilot_class_counts = sample_pilot_data(
    RAW_URL,
    max_per_class=PILOT_SAMPLES_PER_CLASS,
    seed=SEED,
)

total_pilot    = len(pilot_stems)
n_short = sum(1 for n in pilot_class_counts.values() if n < (PILOT_SAMPLES_PER_CLASS or 9999))

print('샘플링 결과')
print(f'  샘플링된 비디오 : {total_pilot}개')
print(f'  클래스 수       : {len(pilot_class_counts)}')
print(f'  데이터가 부족한 클래스 (< {PILOT_SAMPLES_PER_CLASS}개): {n_short}개')

pilot_qa   = [e for e in qa_data if Path(e['video_id']).stem in pilot_stems]
n_qa_pairs = sum(len(e['qa_pairs']) for e in pilot_qa)
print(f'  파일럿 QA 항목  : {len(pilot_qa)}개 비디오 / {n_qa_pairs} QA 쌍')

# 분포 시각화
sorted_class_names = sorted(pilot_class_counts.keys())
y_after = [pilot_class_counts[c] for c in sorted_class_names]
fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(len(sorted_class_names)), y_after, color='steelblue', alpha=0.85, width=1.0)
ax.axhline(PILOT_SAMPLES_PER_CLASS, color='crimson', linestyle='--', lw=1.5,
           label=f'상한={PILOT_SAMPLES_PER_CLASS}')
ax.set_xlabel('클래스 인덱스'); ax.set_ylabel('비디오 수')
ax.set_title(f'파일럿 샘플링 후 클래스별 비디오 수  (seed={SEED})')
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── HTTP 서버에서 파일럿 샘플만 로컬 SSD로 다운로드 ─────────────
# aria2c 병렬 다운로드(8 connections)로 ~5분 내 완료합니다.
# 이미 다운로드된 파일은 건너뜁니다 (--continue=true).
import subprocess, requests as _req

def download_pilot_to_ssd(pilot_stems, raw_url, label_url, qa_json_url, local_root):
    local_root = Path(local_root)
    lines = []

    for cls in list_http_dir(raw_url):
        for fname in list_http_dir(f'{raw_url}/{cls}'):
            if not fname.endswith('.mp4'): continue
            if Path(fname).stem not in pilot_stems: continue
            dst_dir = local_root / 'raw' / cls
            dst_dir.mkdir(parents=True, exist_ok=True)
            if (dst_dir / fname).exists(): continue      # 이어받기 건너뜀
            lines += [f'{raw_url}/{cls}/{fname}', f'  dir={dst_dir}', f'  out={fname}']

        for fname in list_http_dir(f'{label_url}/{cls}'):
            if not fname.endswith('.json'): continue
            if Path(fname).stem not in pilot_stems: continue
            dst_dir = local_root / 'label' / cls
            dst_dir.mkdir(parents=True, exist_ok=True)
            if (dst_dir / fname).exists(): continue
            lines += [f'{label_url}/{cls}/{fname}', f'  dir={dst_dir}', f'  out={fname}']

    # QA JSON — 작은 파일이므로 requests로 직접
    qa_dst = local_root / 'qa_dataset.json'
    if not qa_dst.exists():
        qa_dst.parent.mkdir(parents=True, exist_ok=True)
        qa_dst.write_bytes(_req.get(qa_json_url, timeout=30).content)
        print(f'QA JSON 저장: {qa_dst}')

    if not lines:
        print('이미 모두 다운로드됨 — 건너뜀'); return

    aria2_list = '/tmp/aria2_pilot.txt'
    Path(aria2_list).write_text('\n'.join(lines))
    n_files = len(lines) // 3
    print(f'다운로드 시작: {n_files}개 파일 → {local_root}')

    subprocess.run([
        'aria2c',
        f'--input-file={aria2_list}',
        '--max-concurrent-downloads=8',   # 클래스 단위 병렬
        '--split=4',                       # 파일당 4 chunk
        '--max-connection-per-server=4',
        '--min-split-size=5M',
        '--continue=true',                 # 이어받기
        '--console-log-level=warn',
    ], check=True)
    print(f'다운로드 완료: {n_files}개 파일 → {local_root}')


download_pilot_to_ssd(pilot_stems, RAW_URL, LABEL_URL, QA_JSON_URL, LOCAL_ROOT)


## 학습

- **Phase 0** 실험은 학습을 건너뛰고 바로 평가합니다.
- **Phase 1** 실험은 파일럿 샘플(10개/클래스)로 학습합니다.
- OVERRIDE 딕셔너리에 값을 설정하면 카탈로그 기본값을 덮어씁니다.

In [ ]:
# ── ExperimentConfig 로드 + 오버라이드 적용 ──────────────────
import copy
exp = copy.deepcopy(EXPERIMENT_CATALOG[EXPERIMENT_NAME])

applied_overrides = {k: v for k, v in OVERRIDE.items() if v is not None}

# ExperimentConfig 필드 오버라이드
for field_name, value in applied_overrides.items():
    if hasattr(exp, field_name):
        setattr(exp, field_name, value)
        print(f'  OVERRIDE (ExperimentConfig): {field_name} = {value}')

if not applied_overrides:
    print('  오버라이드 없음 — 카탈로그 기본값 사용')

# ── TrainingConfig 빌드 ───────────────────────────────────────
base_cfg = TrainingConfig(
    qa_json_path      = QA_JSON_PATH,
    raw_video_root    = VIDEO_ROOT,
    label_root        = LABEL_ROOT,
    seed              = SEED,
    max_steps         = 2 if SMOKE_TEST else -1,
    smoke_test        = SMOKE_TEST,
)

training_cfg = exp.build_training_config(
    base_config      = base_cfg,
    checkpoint_root  = CHECKPOINT_ROOT,
)

# TrainingConfig 전용 필드 오버라이드 (ExperimentConfig에 없는 필드)
for field_name, value in applied_overrides.items():
    if not hasattr(exp, field_name):
        if hasattr(training_cfg, field_name):
            setattr(training_cfg, field_name, value)
            print(f'  OVERRIDE (TrainingConfig):  {field_name} = {value}')
        else:
            print(f'  [WARNING] 알 수 없는 필드: {field_name}')

# 파일럿 샘플링 적용
training_cfg.max_samples_per_category = PILOT_SAMPLES_PER_CLASS
training_cfg.video_cache_dir = VIDEO_CACHE_DIR

print('\n최종 TrainingConfig')
print(f'  모델            : {training_cfg.model_id}')
print(f'  LoRA r / α      : {training_cfg.lora_r} / {training_cfg.lora_alpha}')
print(f'  dropout         : {training_cfg.lora_dropout}')
print(f'  learning_rate   : {training_cfg.learning_rate}')
print(f'  fps             : {training_cfg.fps}')
print(f'  max_pixels      : {training_cfg.max_pixels}')
print(f'  grad_accum      : {training_cfg.gradient_accumulation_steps}')
print(f'  num_workers     : {training_cfg.dataloader_num_workers}')
print(f'  에폭            : {training_cfg.num_train_epochs}')
print(f'  eval_steps      : {training_cfg.eval_steps}')
print(f'  save_steps      : {training_cfg.save_steps}')
print(f'  최대 샘플/클래스: {training_cfg.max_samples_per_category}')
print(f'  train/val 비율  : {training_cfg.train_ratio} / {training_cfg.val_ratio}')
print(f'  smoke_test      : {training_cfg.smoke_test}')
print(f'  체크포인트      : {training_cfg.checkpoint_dir}')
print(f'  프레임 캐시     : {training_cfg.video_cache_dir or "(비활성)"}')
print(f'  시스템 프롬프트 : {training_cfg.system_prompt[:60]}...' if len(training_cfg.system_prompt) > 60 else f'  시스템 프롬프트 : {training_cfg.system_prompt or "(없음)"}')

In [ ]:
# ── 학습 실행 ─────────────────────────────────────────────────
from training_runner.train import run_training

trained_adapter_dir = None

if exp.skip_training:
    print(f'[Phase 0] 학습 없음 — 제로샷 베이스라인 평가만 진행합니다.')
else:
    print('학습 시작...')
    t0 = time.time()
    run_training(training_cfg)
    elapsed = time.time() - t0
    trained_adapter_dir = Path(training_cfg.checkpoint_dir) / 'final_adapter'
    print(f'학습 완료  ({elapsed/60:.1f}분)')
    print(f'어댑터 저장 위치: {trained_adapter_dir}')

In [ ]:
# ── (선택) Gradient 흐름 진단 ─────────────────────────────────
# 학습 후 LoRA 레이어별 gradient norm을 확인합니다.
# 실험 설계 문서 4-2절 대응: grad ≈ 0 레이어가 있으면 설정 오류일 수 있습니다.
#
# 정상 범위: 1e-4 ~ 1e-1
# 증상별 원인:
#   모든 LoRA grad=0  → enable_input_require_grads() 누락 또는 순서 오류
#   특정 레이어만 0   → target_modules 설정 오류
#   vision 레이어만 0 → 시각 인코더 frozen (visual encoder LoRA 추가 고려)

# ※ 이 셀은 학습 루프 내에서 model 객체에 접근 가능할 때 실행하세요.
#   run_training() 후에는 모델이 저장되고 메모리에서 해제되므로,
#   아래 진단은 train.py 내부에서 직접 호출하는 것을 권장합니다.

# from training_runner.evaluation.evaluator import diagnose_gradient_flow
# diagnose_gradient_flow(model)   # model이 메모리에 있을 때 실행

print('Gradient 진단은 학습 루프 내에서 model 객체가 있을 때 실행하세요.')
print('training_runner/evaluation/evaluator.py 의 diagnose_gradient_flow() 참고')

## 평가

test split에 대해 모델을 추론하고 question_type별 지표를 계산합니다.

| question_type | 1차 지표 | 2차 지표 |
|---|---|---|
| `fault_ratio` | **Exact Match** | MAE |
| `fault_compare` | **Accuracy** | F1 |
| `accident_place` | ROUGE-L | Exact Match |
| `accident_place_feature` | ROUGE-L | BERTScore |
| `vehicle_a_progress` | ROUGE-L | BERTScore |
| `vehicle_b_progress` | ROUGE-L | BERTScore |

In [ ]:
# ── 모델 로드 (학습 완료 또는 베이스라인) ────────────────────
from peft import PeftModel
from transformers import AutoProcessor
from traffic_ai_core.Qwen3_VL_4B_Instruct.model.model import load_model
from training_runner.dataset import TrafficAccidentQADataset

print('기반 모델 로드 중...')
base_model, processor = load_model(training_cfg.model_id)

adapter_dir = Path(training_cfg.checkpoint_dir) / 'final_adapter'
if adapter_dir.exists() and not exp.skip_training:
    print(f'LoRA 어댑터 로드: {adapter_dir}')
    model = PeftModel.from_pretrained(base_model, str(adapter_dir))
else:
    print('베이스라인 모델 사용 (어댑터 없음)')
    model = base_model

model.eval()

# ── test split 구성 ───────────────────────────────────────────
# 파일럿 샘플링: pilot_stems로 허용 비디오 제한
# (학습과 동일한 max_samples_per_category 사용 → 동일 분할)
full_ds = TrafficAccidentQADataset(
    qa_json_path            = training_cfg.qa_json_path,
    raw_video_root          = training_cfg.raw_video_root,
    label_root              = training_cfg.label_root,
    processor               = processor,
    fps                     = training_cfg.fps,
    max_pixels              = training_cfg.max_pixels,
    question_types          = training_cfg.question_types,
    max_samples_per_category= training_cfg.max_samples_per_category,
    system_prompt           = training_cfg.system_prompt,
)
all_samples = full_ds.get_all_samples()

# case_code별 계층적 분할 — build_dataloaders와 동일한 로직으로 test split 재현
from training_runner.dataset import stratified_split_by_category
_, _, test_ids = stratified_split_by_category(
    all_samples,
    train_ratio=training_cfg.train_ratio,
    val_ratio=training_cfg.val_ratio,
    seed=training_cfg.seed,
)
test_samples = [s for s in all_samples if s.video_id in test_ids]

print(f'\n데이터 분할')
print(f'  전체 비디오: {len(unique_ids)}')
print(f'  train / val / test: {n_train} / {n_val} / {len(test_ids)}')
print(f'  테스트 샘플: {len(test_samples)}개 QA 쌍')

# ── 평가 실행 ────────────────────────────────────────────────
eval_config = EvaluatorConfig(
    fps           = training_cfg.fps,
    max_pixels    = training_cfg.max_pixels,
    system_prompt = training_cfg.system_prompt,
    verbose       = True,
)
aggregated, per_sample_results = run_evaluation(
    model, processor, test_samples,
    config          = eval_config,
    experiment_name = EXPERIMENT_NAME,
)

In [ ]:
# ── 결과 저장 + 시각화 ───────────────────────────────────────
import pandas as pd
from datetime import datetime

# 저장
result_dir = Path(OUTPUT_DIR) / EXPERIMENT_NAME
result_dir.mkdir(parents=True, exist_ok=True)

summary_payload = {
    'experiment'  : EXPERIMENT_NAME,
    'phase'       : exp.phase,
    'description' : exp.description,
    'timestamp'   : datetime.now().isoformat(),
    'config': {
        'lora_r'                  : training_cfg.lora_r,
        'lora_alpha'              : training_cfg.lora_alpha,
        'fps'                     : training_cfg.fps,
        'learning_rate'           : training_cfg.learning_rate,
        'num_train_epochs'        : training_cfg.num_train_epochs,
        'system_prompt_key'       : exp.system_prompt_key,
        'max_samples_per_category': training_cfg.max_samples_per_category,
        'pilot_stems_count'       : len(pilot_stems),
    },
    'metrics': aggregated,
}

with open(result_dir / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary_payload, f, ensure_ascii=False, indent=2)
with open(result_dir / 'per_sample.json', 'w', encoding='utf-8') as f:
    json.dump(per_sample_results, f, ensure_ascii=False, indent=2)
print(f'결과 저장: {result_dir}')

# ── question_type별 지표 테이블 ──────────────────────────────
overall   = aggregated.get('overall', {})
by_qtype  = aggregated.get('by_question_type', {})

rows = []
for qtype, stats in by_qtype.items():
    row = {'question_type': qtype, 'n': stats.get('n_samples', 0)}
    if 'rouge_l_mean' in stats:
        row['ROUGE-L'] = round(stats['rouge_l_mean'], 4)
    if 'exact_match' in stats:
        row['Exact Match'] = round(stats['exact_match'], 4)
    if 'mae_mean' in stats:
        row['MAE'] = round(stats['mae_mean'], 2)
    rows.append(row)

df_results = pd.DataFrame(rows).set_index('question_type')
display(df_results)

# ── 바 차트 ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle(f'실험 [{EXPERIMENT_NAME}] 평가 결과', fontsize=13, fontweight='bold')

# ROUGE-L
rouge_items = [(k, v['rouge_l_mean']) for k, v in by_qtype.items() if 'rouge_l_mean' in v]
if rouge_items:
    labels_r, vals_r = zip(*rouge_items)
    bars_r = axes[0].bar(labels_r, vals_r, color='steelblue', alpha=0.85)
    axes[0].set_ylim(0, 1.0)
    axes[0].set_title('ROUGE-L')
    axes[0].set_ylabel('Score')
    axes[0].tick_params(axis='x', rotation=25)
    for bar, val in zip(bars_r, vals_r):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', fontsize=9)
else:
    axes[0].text(0.5, 0.5, 'ROUGE-L 없음', ha='center', va='center', transform=axes[0].transAxes)

# Exact Match / Accuracy
em_items = [(k, v['exact_match']) for k, v in by_qtype.items() if 'exact_match' in v]
if em_items:
    labels_e, vals_e = zip(*em_items)
    color_e = ['crimson' if k == 'fault_ratio' else 'darkorange' for k in labels_e]
    bars_e  = axes[1].bar(labels_e, vals_e, color=color_e, alpha=0.85)
    axes[1].set_ylim(0, 1.0)
    axes[1].set_title('Exact Match / Accuracy')
    axes[1].set_ylabel('Score')
    axes[1].tick_params(axis='x', rotation=25)
    for bar, val in zip(bars_e, vals_e):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'EM 없음', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
chart_path = result_dir / 'metrics_chart.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'차트 저장: {chart_path}')

# ── 핵심 지표 요약 ────────────────────────────────────────────
def _fmt(v):
    return f'{v:.4f}' if isinstance(v, float) else 'N/A'

print('')
print('=' * 52)
print(f'  핵심 지표 요약  [{EXPERIMENT_NAME}]')
print('=' * 52)
print(f'  fault_ratio  Exact Match : {_fmt(overall.get("fault_ratio_em"))}  ← 핵심')
print(f'  fault_ratio  MAE         : {_fmt(overall.get("fault_ratio_mae_mean"))}')
print(f'  fault_compare Accuracy   : {_fmt(overall.get("fault_compare_acc"))}')
print(f'  ROUGE-L 평균             : {_fmt(overall.get("rouge_l_mean"))}')
print(f'  총 테스트 샘플           : {overall.get("n_samples", "N/A")}')
print('=' * 52)

# 메모리 해제
del model, base_model
torch.cuda.empty_cache()
print('GPU 메모리 해제 완료')

In [ ]:
# ── 학습 결과를 Drive로 백업 ─────────────────────────────────────
# 로컬 체크포인트 중 final_adapter만 Drive로 rsync합니다.
# 세션이 끊겨도 어댑터가 Drive에 보존됩니다.
import subprocess

adapter_local = f'/content/checkpoints/{exp.checkpoint_subdir}/final_adapter'
adapter_drive = f'{DRIVE_ROOT}/checkpoints/{exp.checkpoint_subdir}/final_adapter'

if Path(adapter_local).exists():
    Path(adapter_drive).mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ['rsync', '-a', '--info=progress2', adapter_local + '/', adapter_drive + '/'],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f'어댑터 백업 완료: {adapter_drive}')
    else:
        print(f'rsync 오류:\n{result.stderr}')
else:
    print(f'어댑터 없음 (Phase 0 또는 학습 실패): {adapter_local}')


In [ ]:
# ── (선택) 완료된 실험 비교 테이블 ───────────────────────────
# 여러 실험을 순서대로 실행한 뒤 이 셀을 실행하면 결과를 한눈에 비교할 수 있습니다.

results_root = Path(OUTPUT_DIR)
completed = sorted(results_root.rglob('summary.json'))

if not completed:
    print('완료된 실험 없음 — 실험을 먼저 실행하세요.')
else:
    compare_rows = []
    for sp in completed:
        with open(sp, encoding='utf-8') as f:
            s = json.load(f)
        ov  = s.get('metrics', {}).get('overall', {})
        cfg = s.get('config', {})
        compare_rows.append({
            'experiment'    : s.get('experiment', ''),
            'phase'         : s.get('phase', ''),
            'fault_ratio_EM': ov.get('fault_ratio_em'),
            'ROUGE-L_mean'  : ov.get('rouge_l_mean'),
            'compare_Acc'   : ov.get('fault_compare_acc'),
            'lora_r'        : cfg.get('lora_r'),
            'fps'           : cfg.get('fps'),
            'lr'            : cfg.get('learning_rate'),
            'prompt'        : cfg.get('system_prompt_key'),
        })

    df_compare = pd.DataFrame(compare_rows).sort_values(['phase', 'experiment'])
    df_compare = df_compare.set_index('experiment')

    # 수치 포맷
    for col in ['fault_ratio_EM', 'ROUGE-L_mean', 'compare_Acc']:
        df_compare[col] = df_compare[col].map(lambda v: f'{v:.4f}' if v is not None else 'N/A')

    print(f'완료된 실험: {len(compare_rows)}개')
    display(df_compare)

    # fault_ratio EM 막대 차트
    numeric_rows = [
        r for r in compare_rows
        if isinstance(r.get('fault_ratio_EM'), float)
    ]
    if numeric_rows:
        exp_names = [r['experiment'] for r in numeric_rows]
        em_vals   = [r['fault_ratio_EM'] for r in numeric_rows]
        fig, ax   = plt.subplots(figsize=(max(8, len(exp_names) * 1.2), 4))
        colors_c  = ['crimson' if n == EXPERIMENT_NAME else 'steelblue' for n in exp_names]
        ax.bar(exp_names, em_vals, color=colors_c, alpha=0.85)
        ax.set_ylim(0, 1.0)
        ax.set_title('fault_ratio Exact Match 비교 (핵심 지표)')
        ax.set_ylabel('Exact Match')
        ax.tick_params(axis='x', rotation=30)
        plt.tight_layout()
        plt.show()